# Plan Step 4: NN eval 学習 (Colab T4 GPU)

**目的**: ローカル WSL の CPU torch では現実的でない NN 学習 (= Step 4) を Colab の無料 GPU で代行する。

**入力**: `db/self_play_snapshots.jsonl` (= Step 2 で蓄積した約 30 万 snapshot、 695MB)。

**出力**: `db/nn_eval.pt` + `db/_nn_feature_keys.json` (= ローカルに戻して Step 5 で使う)。

## 使い方

1. **Drive 準備**: Google Drive のルートに `onepiece_nn/` フォルダを作り、 `self_play_snapshots.jsonl` をアップロード
2. **Colab 起動**: この notebook を Colab で開く (= File > Upload notebook)
3. **GPU 切替**: ランタイム > ランタイムのタイプを変更 > T4 GPU
4. **全セル実行**: ランタイム > すべてのセルを実行
5. **成果物 DL**: 完了後 Drive の `onepiece_nn/nn_eval.pt` + `onepiece_nn/_nn_feature_keys.json` をローカルの `db/` に置く

T4 GPU で 30 万 snapshot × 20 epoch = 約 5-15 分の見込み。

## 1. GPU 確認

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
assert torch.cuda.is_available(), 'GPU が有効化されていません。 ランタイム > ランタイムのタイプを変更 > T4 GPU を選択してください。'

## 2. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, 'self_play_snapshots.jsonl')
OUTPUT_PT = os.path.join(WORK_DIR, 'nn_eval.pt')
OUTPUT_KEYS = os.path.join(WORK_DIR, '_nn_feature_keys.json')

assert os.path.exists(SNAPSHOT_PATH), f'snapshot が見つからない: {SNAPSHOT_PATH}\nDrive ルートに onepiece_nn/ フォルダを作り、 self_play_snapshots.jsonl をアップロードしてください。'
print('snapshot size:', os.path.getsize(SNAPSHOT_PATH) // (1024*1024), 'MB')

## 3. snapshot ロード

30 万行の jsonl を 78 dim float tensor + ±1 winner label に変換。 メモリは 約 100MB 程度。

In [ ]:
import json
import time

# action category 数 (= engine.nn_eval.N_ACTION_CATEGORIES と同期、 現状 9)
N_ACTION_CATEGORIES = 9

t0 = time.time()
snapshots = []
feature_keys = None
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        snapshots.append(snap)
        if feature_keys is None and 'features' in snap:
            feature_keys = sorted(snap['features'].keys())

print(f'loaded {len(snapshots):,} snapshots in {time.time()-t0:.1f}s')
print(f'feature_keys dim: {len(feature_keys)}')

# outcome 分布
win = sum(1 for s in snapshots if s.get('winner', s.get('final_winner', 0)) > 0)
lose = sum(1 for s in snapshots if s.get('winner', s.get('final_winner', 0)) < 0)
draw = sum(1 for s in snapshots if s.get('winner', s.get('final_winner', 0)) == 0)
print(f'outcome dist: win={win:,}, lose={lose:,}, draw={draw:,}')

## 4. Tensor 化

In [ ]:
import numpy as np

device = torch.device('cuda')

t0 = time.time()
X_np = np.zeros((len(snapshots), len(feature_keys)), dtype=np.float32)
y_np = np.zeros((len(snapshots),), dtype=np.float32)
for i, snap in enumerate(snapshots):
    feats = snap.get('features', {})
    for j, k in enumerate(feature_keys):
        X_np[i, j] = float(feats.get(k, 0.0))
    y_np[i] = float(snap.get('winner', snap.get('final_winner', 0)))
print(f'tensor build: {time.time()-t0:.1f}s')
print(f'X shape: {X_np.shape}, y shape: {y_np.shape}')

X = torch.from_numpy(X_np).to(device)
y = torch.from_numpy(y_np).unsqueeze(1).to(device)
a = torch.zeros(len(snapshots), dtype=torch.long, device=device)  # action_idx fallback (snapshot に無いため)

# 80/20 split
n = len(snapshots)
split = int(n * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
a_train, a_val = a[:split], a[split:]
print(f'train: {X_train.shape[0]:,}, val: {X_val.shape[0]:,}')

## 5. モデル定義

`scripts/train_nn_eval.py:SimpleEvaluator` と同じ構造 (= 78 → 64 → 32 → value/policy)。

In [ ]:
import torch.nn as nn

class SimpleEvaluator(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.value_head = nn.Linear(hidden_dim // 2, 1)
        self.policy_head = nn.Linear(hidden_dim // 2, N_ACTION_CATEGORIES)

    def forward(self, x):
        h = self.shared(x)
        return self.value_head(h), self.policy_head(h)

model = SimpleEvaluator(input_dim=len(feature_keys)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params: {n_params:,}')

## 6. 学習

20 epoch、 batch 512 (= T4 GPU なので大きめ)、 lr 1e-3。

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 20
BATCH = 512
LR = 1e-3

train_ds = TensorDataset(X_train, y_train, a_train)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LR)
value_criterion = nn.MSELoss()
policy_criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    n_batches = 0
    for Xb, yb, ab in train_loader:
        optimizer.zero_grad()
        v_pred, p_pred = model(Xb)
        loss_v = value_criterion(v_pred, yb)
        loss_p = policy_criterion(p_pred, ab)
        loss = loss_v + 0.5 * loss_p
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1

    model.eval()
    with torch.no_grad():
        v_pred_val, _ = model(X_val)
        sign_acc = float((torch.sign(v_pred_val) == torch.sign(y_val)).float().mean().item())

    avg_loss = train_loss_sum / max(1, n_batches)
    print(f'epoch {epoch+1:2d}/{EPOCHS}: train_loss={avg_loss:.4f}, val_sign_acc={sign_acc:.3f}')

    if sign_acc > best_val_acc:
        best_val_acc = sign_acc
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, OUTPUT_PT)
        with open(OUTPUT_KEYS, 'w', encoding='utf-8') as f:
            json.dump(feature_keys, f, ensure_ascii=False)
        print(f'    ✔ saved {OUTPUT_PT} (best val_sign_acc={sign_acc:.3f})')

print(f'\n=== DONE. best val_sign_acc={best_val_acc:.3f} ===')

## 7. 成果物の確認

Drive に `nn_eval.pt` + `_nn_feature_keys.json` が保存されているはず。 これをローカルにダウンロードして `db/` 配下に置けば、 ローカル engine の `engine/nn_eval.py` が auto-load する。

In [ ]:
import os
print('--- 成果物 ---')
for p in [OUTPUT_PT, OUTPUT_KEYS]:
    if os.path.exists(p):
        print(f'  {p}: {os.path.getsize(p):,} bytes')
    else:
        print(f'  {p}: NOT FOUND')

print('\n--- 次の手順 ---')
print('1. Drive の onepiece_nn/nn_eval.pt と onepiece_nn/_nn_feature_keys.json をローカルにダウンロード')
print('2. ~/projects/onepiece_research/db/ 配下に置く (= 既存 nn_eval.pt は上書き)')
print('3. ローカルで .venv/bin/python -c "from engine.nn_eval import get_nn_evaluator; e=get_nn_evaluator(); print(e is not None)" で読込確認')